# Notebook 6: Manifold mapping for discovery and steering

This notebook focuses on the manifold sidecar as an advisory discovery layer: it ranks inference candidates and facet relations using geometry-aware scores so an agent can prioritize which neighborhoods to inspect next.

In [ ]:
from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find the repository root.")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def show(data):
    if isinstance(data, str):
        print(data)
    else:
        print(json.dumps(data, indent=2, default=str))


In [ ]:
from src.inference.models import FacetRelation, InferenceNode
from src.manifold_sidecar import ManifoldRankingRequest, ManifoldRankingService

memory_root = REPO_ROOT / "notebooks" / ".demo-memory-06"
shutil.rmtree(memory_root, ignore_errors=True)
service = ManifoldRankingService(
    model_id="manifold-discovery-v1",
    geometry_version="graph-neighborhood-geometry-v2",
    embedding_version="text-graph-temporal-v2",
)
show(service.get_current_model_metadata().model_dump(mode="json"))

In [ ]:
inference_candidates = [
    InferenceNode(
        text="Inference candidate: The certificate rollback is the strongest explanation for the recovery.",
        inference_mode="abductive",
        confidence=0.86,
        assumptions=["Recovery followed the rollback quickly"],
        generated_from_nodes=["claim-a", "claim-b"],
        generated_from_edges=["timeline:rollback", "timeline:recovery"],
        source_branch="inference/discovery-map",
        source_commit="seed-commit-1",
        source_session="discovery-map",
        prompt_template_id="inference-template-v1",
        policy_version="policy-v1",
        retrieval_context_ids=["claim-a", "claim-b"],
        parent_nodes=["claim-a", "claim-b"],
        parent_edges=["timeline:rollback", "timeline:recovery"],
    ),
    InferenceNode(
        text="Inference candidate: Refund failures may also have been amplified by a backlog in the retry queue.",
        inference_mode="abductive",
        confidence=0.61,
        assumptions=["Queue pressure was elevated during the incident"],
        generated_from_nodes=["claim-c"],
        generated_from_edges=["queue:pressure"],
        source_branch="inference/discovery-map",
        source_commit="seed-commit-1",
        source_session="discovery-map",
        prompt_template_id="inference-template-v1",
        policy_version="policy-v1",
        retrieval_context_ids=["claim-c"],
        parent_nodes=["claim-c"],
        parent_edges=["queue:pressure"],
    ),
]

facet_candidates = [
    FacetRelation(
        source_node_id="claim-a",
        target_node_id="claim-b",
        facet_type="same_claim_different_timeframe",
        same_proposition=False,
        shared_core_claim="Acme Payments refund processing",
        differences=["yesterday", "today"],
        provenance_commit="seed-commit-1",
        source_branch="inference/discovery-map",
        ranking_model_id="pending-ranking",
        ranking_run_id="pending-ranking",
        geometry_version=None,
        relatedness_score=0.0,
        distance_score=1.0,
        facet_strength=0.0,
        prompt_template_id="inference-template-v1",
        policy_version="policy-v1",
        parent_nodes=["claim-a", "claim-b"],
        parent_edges=["timeline:incident", "timeline:recovery"],
    ),
    FacetRelation(
        source_node_id="claim-b",
        target_node_id="claim-c",
        facet_type="same_claim_different_scope",
        same_proposition=False,
        shared_core_claim="Acme Payments retry handling",
        differences=["enterprise customers", "all customers"],
        provenance_commit="seed-commit-1",
        source_branch="inference/discovery-map",
        ranking_model_id="pending-ranking",
        ranking_run_id="pending-ranking",
        geometry_version=None,
        relatedness_score=0.0,
        distance_score=1.0,
        facet_strength=0.0,
        prompt_template_id="inference-template-v1",
        policy_version="policy-v1",
        parent_nodes=["claim-b", "claim-c"],
        parent_edges=["scope:enterprise", "scope:global"],
    ),
]

In [ ]:
ranked_inference = service.rank_inference_candidates(
    ManifoldRankingRequest(
        branch_name="inference/discovery-map",
        ranking_mode="inference_candidate_ranking",
        seed_context={"goal": "find the most promising explanation path"},
        candidates=inference_candidates,
    )
)
ranked_facets = service.rank_facet_candidates(
    ManifoldRankingRequest(
        branch_name="inference/discovery-map",
        ranking_mode="facet_candidate_ranking",
        seed_context={"goal": "find bridges between related memories"},
        candidates=facet_candidates,
    )
)
show({
    "ranked_inference": [candidate.model_dump(mode="json") for candidate in ranked_inference.candidates],
    "ranked_facets": [candidate.model_dump(mode="json") for candidate in ranked_facets.candidates],
})

In [ ]:
discovery_queue = [
    {
        "kind": "inference",
        "text": candidate.text,
        "priority": candidate.ranking_score,
        "distance": candidate.distance_score,
        "relatedness": candidate.relatedness_score,
    }
    for candidate in ranked_inference.candidates
] + [
    {
        "kind": "facet",
        "text": candidate.shared_core_claim,
        "priority": candidate.facet_strength,
        "distance": candidate.distance_score,
        "relatedness": candidate.relatedness_score,
    }
    for candidate in ranked_facets.candidates
]
discovery_queue.sort(key=lambda item: item["priority"], reverse=True)
show({
    "geometry_version": service.geometry_version,
    "model_id": service.model_id,
    "discovery_queue": discovery_queue,
})

## What this notebook demonstrates

- manifold metadata describing the active ranking model and geometry
- geometric scoring of inference candidates for discovery prioritization
- geometric scoring of facet relations for bridge discovery
- advisory ranking signals that inform steering without replacing the graph as source of truth